In [ ]:
!pip install -q transformers accelerate safetensors

import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from google.colab import files
from transformers import pipeline

uploaded = files.upload()
image_paths = list(uploaded.keys())

if len(image_paths) != 5:
    raise ValueError("Please upload exactly 5 mug images.")

device = 0 if torch.cuda.is_available() else -1

depth_pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=device
)

results = []

for image_path in image_paths:
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    result = depth_pipe(image)
    depth = np.array(result["depth"]).astype(np.float32)

    depth = cv2.resize(depth, image.size)
    depth_norm = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    depth_vis = (depth_norm * 255).astype(np.uint8)
    depth_color = cv2.applyColorMap(depth_vis, cv2.COLORMAP_INFERNO)
    depth_color = cv2.cvtColor(depth_color, cv2.COLOR_BGR2RGB)

    results.append((image_np, depth_color))

fig, axes = plt.subplots(5, 2, figsize=(10, 25))

for i, (image_np, depth_color) in enumerate(results):
    axes[i, 0].imshow(image_np)
    axes[i, 0].axis("off")
    axes[i, 0].set_title(f"Input Mug Image {i+1}")

    axes[i, 1].imshow(depth_color)
    axes[i, 1].axis("off")
    axes[i, 1].set_title(f"Depth Anything V2 Depth Map {i+1}")

plt.tight_layout()

output_path = "depth_anything_v2_depth_maps_5_mugs.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

files.download(output_path)